# Producto 2 · Inferencia del Modelo y Score de Decisión Corporativa (`0–100`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Darkops56/CapstoneProject/blob/main/producto2/inference.ipynb)

Este notebook presenta **paso a paso** cómo cargar los artefactos entrenados (`.joblib`) y probar el sistema productivo de **inferencia inmobiliaria** del Producto 2 en CABA.

### Qué evalúa el sistema
1. **Segmentación Urbana (`K-Means`, $k=5$):** asigna a la propiedad un perfil de habitabilidad urbana (ej. *Zona Familiar Tranquila*, *Hub Ejecutivo Conectado*, etc.).
2. **Precio Justo Teórico (`XGBoost Regressor`):** estima el `precio_justo_ars` mensual (alquiler + expensas) a partir de las **10 features ortogonales** del Producto 1.
3. **Detección de Sobreprecio / Ahorro (`%`):** compara el precio publicado contra el `precio_justo_ars`.
4. **Score de Decisión (`0–100`) y Veredicto Automático:** puntúa cada propiedad según el perfil del empleado (**`FAMILIA`** o **`SOLTERO_EJECUTIVO`**).

### Mapa de Pasos
0. **Configuración del Entorno (Local y Google Colab)**.
1. **Carga e Inspección de Artefactos `.joblib`** (`kmeans_model`, `xgb_price_model`, `model_metadata`).
2. **Fórmula del Score de Decisión (`0–100`) y Funciones de Inferencia**.
3. **Prueba 1 · Inferencia Individual (`predict_property`) + Simulador Interactivo**.
4. **Prueba 2 · Inferencia Batch (`predict_batch`) sobre las 964 propiedades de CABA**.
5. **Visualización de Resultados**: Precio Publicado vs. Precio Justo y Ranking por Perfil.
6. **Salida JSON / Exportación** lista para integración con Make/n8n o Dashboard.

## 0. Configuración del Entorno (Local y Google Colab)

Detecta automáticamente si se ejecuta en **Google Colab** o de forma **local**, instala librerías faltantes si fuera necesario (`xgboost`, `scikit-learn`, `joblib`, `seaborn`, `matplotlib`) y localiza los artefactos `.joblib` y el dataset de prueba.

In [ ]:
import importlib
import json
import subprocess
import sys
from pathlib import Path
from typing import Any, Literal

IN_COLAB = "google.colab" in sys.modules

REQUIRED_PACKAGES = {
    "joblib": "joblib",
    "xgboost": "xgboost",
    "sklearn": "scikit-learn",
    "seaborn": "seaborn",
    "matplotlib": "matplotlib",
}
missing_pkgs = [pip_name for mod, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    print(f"Instalando dependencias faltantes: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

REPO_URL = "https://github.com/Darkops56/CapstoneProject.git"

def resolve_project_root() -> Path:
    for cand in [Path.cwd(), Path.cwd().parent, Path("/content/CapstoneProject")]:
        if (cand / "producto2" / "artifacts").exists():
            return cand.resolve()
    if IN_COLAB:
        target = Path("/content/CapstoneProject")
        print(f"Clonando repositorio en {target}...")
        res = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], capture_output=True, text=True)
        if res.returncode == 0 and (target / "producto2" / "artifacts").exists():
            return target.resolve()
        print("No se pudo clonar automáticamente. Puedes subir los archivos .joblib y el CSV manualmente.")
    return Path.cwd().resolve()

BASE_DIR = resolve_project_root()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

PRODUCTO1_DIR = BASE_DIR / "producto1"
PRODUCTO2_DIR = BASE_DIR / "producto2"
ARTIFACTS_DIR = PRODUCTO2_DIR / "artifacts"

# En Colab sin repo clonado, permitir subir artefactos manualmente si faltan
req_artifact_names = ["kmeans_model.joblib", "xgb_price_model.joblib", "model_metadata.joblib"]
if IN_COLAB and not all((ARTIFACTS_DIR / name).exists() for name in req_artifact_names):
    if all((Path.cwd() / name).exists() for name in req_artifact_names):
        ARTIFACTS_DIR = Path.cwd()
    else:
        from google.colab import files
        print("Sube los 3 archivos .joblib (kmeans_model.joblib, xgb_price_model.joblib, model_metadata.joblib):")
        files.upload()
        ARTIFACTS_DIR = Path.cwd()

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'} | BASE_DIR: {BASE_DIR}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")
print(f"Versiones -> xgboost: {xgboost.__version__} | scikit-learn: {sklearn.__version__} | joblib: {joblib.__version__}")
for name in req_artifact_names:
    print(f"  - {name:24s}: {'OK' if (ARTIFACTS_DIR / name).exists() else 'FALTA'}")

## 1. Carga e Inspección de Artefactos Entrenados (`.joblib`)

Cargamos los tres archivos serializados en `producto2/artifacts/`:
- `kmeans_model.joblib`: `Pipeline(StandardScaler → KMeans)` para segmentar entornos urbanos.
- `xgb_price_model.joblib`: `TransformedTargetRegressor(XGBRegressor)` entrenado con `log1p` / `expm1`.
- `model_metadata.joblib`: diccionario con las 10 features de regresión, las features de clustering y las etiquetas descriptivas de cada clúster.

In [ ]:
def load_artifacts(artifacts_dir: Path | str = ARTIFACTS_DIR) -> dict[str, Any]:
    adir = Path(artifacts_dir)
    return {
        "kmeans_pipeline": joblib.load(adir / "kmeans_model.joblib"),
        "xgb_model": joblib.load(adir / "xgb_price_model.joblib"),
        "metadata": joblib.load(adir / "model_metadata.joblib"),
    }

artifacts = load_artifacts(ARTIFACTS_DIR)
meta = artifacts["metadata"]

print("Target:", meta.get("target_col"))
print("\n10 Features requeridas para regresión (XGBoost):")
for i, feat in enumerate(meta["regression_features"], 1):
    print(f"  {i:2d}. {feat}")

print("\nFeatures de entorno para segmentación (K-Means):")
print(" ", meta["cluster_features"])

print("\nEtiquetas de Clústeres de Habitabilidad:")
display(pd.DataFrame.from_dict(meta["cluster_labels"], orient="index", columns=["cluster_label"]).rename_axis("cluster_id"))

## 2. Fórmula del Score de Decisión (`0–100`) y Funciones de Inferencia

Implementamos exactamente la lógica productiva de `producto2/inference.py`:

$$\text{Score} = \text{clip}\big(\text{Score}_{\text{Económico (0–40)}} + \text{Score}_{\text{Clúster (0–25)}} + \text{Score}_{\text{Perfil (0–35)}},\ 0,\ 100\big)$$

- **Económico (`0–40 pts`):** compara `precio_publicado` vs. `precio_justo_ars` (recortado en $[-25\%, +25\%]$; a precio justo $= 20$ pts).
- **Clúster (`0–25 pts`):** premia `"Zona Familiar Tranquila"` para **`FAMILIA`** y `"Hub Ejecutivo Conectado"` / `"Conectada"` para **`SOLTERO_EJECUTIVO`**.
- **Atributos del Perfil (`0–35 pts`):**
  - **`FAMILIA`**: amplitud (`ambientes`), `score_amenities`, `apto_mascotas`, `cocheras`, calma acústica (`ruido_db_promedio`) y cercanía escolar (`distancia_escuela_m`).
  - **`SOLTERO_EJECUTIVO`**: `score_amenities`, conectividad de subte (`lineas_subte_600m`), tipología ágil 1–3 ambientes, `apto_mascotas` y `cocheras`.

In [ ]:
ProfileType = Literal["FAMILIA", "SOLTERO_EJECUTIVO"]

REQUIRED_10_FEATURES = [
    "apto_mascotas", "score_amenities", "m2_totales", "ruido_db_promedio",
    "densidad_delitos_500m", "distancia_escuela_m", "lineas_subte_600m",
    "banos", "ambientes", "cocheras",
]


def compute_decision_score(
    features: dict[str, float] | pd.Series,
    cluster_profile_tag: str,
    pct_ahorro: float,
    profile: ProfileType = "FAMILIA",
) -> float:
    profile_norm = str(profile).strip().upper()
    if profile_norm not in {"FAMILIA", "SOLTERO_EJECUTIVO"}:
        raise ValueError(f"Perfil no soportado: {profile}. Use 'FAMILIA' o 'SOLTERO_EJECUTIVO'.")

    # 1. Componente económico (0..40 pts)
    ahorro_clipped = float(np.clip(pct_ahorro, -25.0, 25.0))
    score_economico = ((ahorro_clipped + 25.0) / 50.0) * 40.0

    # 2. Componente clúster de habitabilidad (0..25 pts)
    tag = cluster_profile_tag.lower()
    if profile_norm == "FAMILIA":
        if "familiar" in tag or "tranquil" in tag:
            score_cluster = 25.0
        elif "conectado" in tag or "equilibrad" in tag:
            score_cluster = 16.0
        else:
            score_cluster = 8.0
    else:
        if "conectado" in tag or "ejecutivo" in tag or "hub" in tag:
            score_cluster = 25.0
        elif "equilibrad" in tag or "familiar" in tag or "tranquil" in tag:
            score_cluster = 16.0
        else:
            score_cluster = 10.0

    # 3. Componente adecuación física + mascotas + amenities (0..35 pts)
    amenities = float(np.clip(float(features.get("score_amenities", 0.0)) / 10.0, 0.0, 1.0))
    mascotas = 1.0 if float(features.get("apto_mascotas", 0.0)) >= 1.0 else 0.0
    cocheras = 1.0 if float(features.get("cocheras", 0.0)) >= 1.0 else 0.0
    ambientes = float(features.get("ambientes", 1.0))
    ruido = float(features.get("ruido_db_promedio", 70.0))
    lineas_subte = float(features.get("lineas_subte_600m", 0.0))
    dist_escuela = float(features.get("distancia_escuela_m", 300.0))

    if profile_norm == "FAMILIA":
        espacio_fit = float(np.clip((ambientes - 1.0) / 3.0, 0.0, 1.0))
        calma_fit = float(np.clip((75.0 - ruido) / 15.0, 0.0, 1.0))
        escuela_fit = float(np.clip(1.0 - (dist_escuela / 600.0), 0.0, 1.0))
        score_perfil = 10.0 * espacio_fit + 7.0 * amenities + 6.0 * mascotas + 5.0 * cocheras + 4.0 * calma_fit + 3.0 * escuela_fit
    else:
        subte_fit = float(np.clip(lineas_subte / 2.0, 0.0, 1.0))
        agil_fit = 1.0 if 1.0 <= ambientes <= 3.0 else 0.6
        score_perfil = 12.0 * amenities + 11.0 * subte_fit + 6.0 * agil_fit + 4.0 * mascotas + 2.0 * cocheras

    return round(float(np.clip(score_economico + score_cluster + score_perfil, 0.0, 100.0)), 2)


def classify_recommendation(decision_score: float, pct_sobreprecio: float) -> str:
    if decision_score >= 75.0 and pct_sobreprecio <= 5.0:
        return "ALTAMENTE RECOMENDADA"
    if decision_score >= 55.0 and pct_sobreprecio <= 15.0:
        return "RECOMENDADA"
    if pct_sobreprecio > 20.0:
        return "DESCARTAR / SOBREPRECIO ALTO"
    return "EVALUAR / ALTERNATIVA"


def predict_batch(
    df: pd.DataFrame,
    profile: ProfileType = "FAMILIA",
    published_price_col: str | None = "costo_total_ars",
    artifacts_dict: dict[str, Any] = artifacts,
) -> pd.DataFrame:
    meta_local = artifacts_dict["metadata"]
    reg_features = meta_local["regression_features"]
    cluster_features = meta_local["cluster_features"]
    cluster_labels = meta_local["cluster_labels"]

    missing_reg = [c for c in reg_features if c not in df.columns]
    if missing_reg:
        raise ValueError(f"Faltan features requeridas para inferencia: {missing_reg}")

    X_reg = df[reg_features].astype(float)
    X_clu = df[cluster_features].astype(float)

    cluster_ids = artifacts_dict["kmeans_pipeline"].predict(X_clu)
    precio_justo = np.clip(artifacts_dict["xgb_model"].predict(X_reg), 1.0, None)

    out = df.copy()
    out["cluster_id"] = cluster_ids.astype(int)
    out["cluster_label"] = [cluster_labels[int(cid)] for cid in cluster_ids]
    out["precio_justo_ars"] = np.round(precio_justo, 2)

    if published_price_col and published_price_col in out.columns:
        pub = out[published_price_col].astype(float).to_numpy()
        pct_sobreprecio = ((pub - precio_justo) / precio_justo) * 100.0
        pct_ahorro = -pct_sobreprecio
    else:
        pct_sobreprecio = np.zeros(len(out), dtype=float)
        pct_ahorro = np.zeros(len(out), dtype=float)

    out["pct_sobreprecio"] = np.round(pct_sobreprecio, 2)
    out["pct_ahorro"] = np.round(pct_ahorro, 2)
    out["decision_score_0_100"] = [
        compute_decision_score(row, str(row["cluster_label"]), float(row["pct_ahorro"]), profile=profile)
        for _, row in out.iterrows()
    ]
    out["recomendacion"] = [
        classify_recommendation(sc, sp) for sc, sp in zip(out["decision_score_0_100"], out["pct_sobreprecio"])
    ]
    return out


def predict_property(
    features: dict[str, float],
    profile: ProfileType = "FAMILIA",
    precio_publicado_ars: float | None = None,
    artifacts_dict: dict[str, Any] = artifacts,
) -> dict[str, Any]:
    row = dict(features)
    pub_col = None
    if precio_publicado_ars is not None:
        row["costo_total_ars"] = float(precio_publicado_ars)
        pub_col = "costo_total_ars"

    rec = predict_batch(pd.DataFrame([row]), profile=profile, published_price_col=pub_col, artifacts_dict=artifacts_dict).iloc[0]
    return {
        "profile": profile,
        "cluster_id": int(rec["cluster_id"]),
        "cluster_label": str(rec["cluster_label"]),
        "precio_justo_ars": float(rec["precio_justo_ars"]),
        "pct_sobreprecio": float(rec["pct_sobreprecio"]),
        "pct_ahorro": float(rec["pct_ahorro"]),
        "decision_score_0_100": float(rec["decision_score_0_100"]),
        "recomendacion": str(rec["recomendacion"]),
    }

print("Funciones de scoring e inferencia listas.")

## 3. Prueba 1 · Inferencia Individual (`predict_property`) + Simulador Interactivo

### 3.1 Comparación de Dos Casos Test en Ambos Perfiles
Evaluamos dos propiedades típicas frente a frente para verificar cómo el modelo distingue **precio justo**, **segmento urbano** y **score por perfil**:
- **Caso A (Depto Familiar Tranquilo):** 95 m², 4 amb., 2 baños, cochera, apto mascotas, ruido `62.5 dBA`, publicado a `\$1,650,000 ARS`.
- **Caso B (Depto Ejecutivo Conectado):** 48 m², 2 amb., 1 baño, amenities `8.0`, 2 líneas de subte a <600 m, publicado a `\$1,550,000 ARS`.

In [ ]:
caso_familiar = {
    "apto_mascotas": 1, "score_amenities": 6.5, "m2_totales": 95.0,
    "ruido_db_promedio": 62.5, "densidad_delitos_500m": 420.0,
    "distancia_escuela_m": 140.0, "lineas_subte_600m": 1.0,
    "banos": 2.0, "ambientes": 4.0, "cocheras": 1.0,
}
caso_ejecutivo = {
    "apto_mascotas": 0, "score_amenities": 8.0, "m2_totales": 48.0,
    "ruido_db_promedio": 72.5, "densidad_delitos_500m": 1100.0,
    "distancia_escuela_m": 280.0, "lineas_subte_600m": 2.0,
    "banos": 1.0, "ambientes": 2.0, "cocheras": 0.0,
}

resultados_demo = [
    {"caso": "Caso A (Familiar 4 amb.)", "precio_publicado_ars": 1_650_000.0, **predict_property(caso_familiar, profile="FAMILIA", precio_publicado_ars=1_650_000.0)},
    {"caso": "Caso A (Familiar 4 amb.)", "precio_publicado_ars": 1_650_000.0, **predict_property(caso_familiar, profile="SOLTERO_EJECUTIVO", precio_publicado_ars=1_650_000.0)},
    {"caso": "Caso B (Ejecutivo 2 amb.)", "precio_publicado_ars": 1_550_000.0, **predict_property(caso_ejecutivo, profile="FAMILIA", precio_publicado_ars=1_550_000.0)},
    {"caso": "Caso B (Ejecutivo 2 amb.)", "precio_publicado_ars": 1_550_000.0, **predict_property(caso_ejecutivo, profile="SOLTERO_EJECUTIVO", precio_publicado_ars=1_550_000.0)},
]

display(pd.DataFrame(resultados_demo)[["caso", "profile", "cluster_label", "precio_publicado_ars", "precio_justo_ars", "pct_sobreprecio", "pct_ahorro", "decision_score_0_100", "recomendacion"]])

### 3.2 Simulador Interactivo en Vivo (Formulario compatible con Google Colab)
Modifica los valores de entrada (en Google Colab aparecen como controles interactivos a la derecha) y vuelve a ejecutar la celda para probar cualquier departamento en vivo durante la presentación.

In [ ]:
# @title Parámetros de la propiedad a evaluar { run: "auto" }
profile = "FAMILIA"  # @param ["FAMILIA", "SOLTERO_EJECUTIVO"]
precio_publicado_ars = 1350000  # @param {type:"number"}
m2_totales = 75  # @param {type:"number"}
ambientes = 3  # @param {type:"integer"}
banos = 2  # @param {type:"integer"}
cocheras = 1  # @param {type:"integer"}
apto_mascotas = 1  # @param [0, 1] {type:"raw"}
score_amenities = 6.0  # @param {type:"slider", min:0, max:10, step:0.5}
ruido_db_promedio = 62.5  # @param {type:"number"}
densidad_delitos_500m = 500  # @param {type:"number"}
distancia_escuela_m = 180  # @param {type:"number"}
lineas_subte_600m = 1  # @param {type:"integer"}

prop_custom = {
    "apto_mascotas": float(apto_mascotas),
    "score_amenities": float(score_amenities),
    "m2_totales": float(m2_totales),
    "ruido_db_promedio": float(ruido_db_promedio),
    "densidad_delitos_500m": float(densidad_delitos_500m),
    "distancia_escuela_m": float(distancia_escuela_m),
    "lineas_subte_600m": float(lineas_subte_600m),
    "banos": float(banos),
    "ambientes": float(ambientes),
    "cocheras": float(cocheras),
}

veredicto = predict_property(prop_custom, profile=profile, precio_publicado_ars=float(precio_publicado_ars))
print("=== VEREDICTO EN VIVO ===")
print(json.dumps(veredicto, ensure_ascii=False, indent=2))

## 4. Prueba 2 · Inferencia Batch (`predict_batch`) sobre el Mercado Completo de CABA

Evaluamos el dataset curado completo (**`964` propiedades**) para ambos perfiles (`FAMILIA` y `SOLTERO_EJECUTIVO`) e identificamos las mejores oportunidades de relocalización corporativa.

In [ ]:
data_candidates = [
    PRODUCTO1_DIR / "producto1_dataset_curado.csv",
    PRODUCTO1_DIR / "dataset_curado.csv",
    PRODUCTO1_DIR / "dataset_enriquecido_completo.csv",
    Path.cwd() / "producto1_dataset_curado.csv",
]

df_input = None
loaded_csv = None
for p in data_candidates:
    if p.exists():
        tmp = pd.read_csv(p)
        if set(REQUIRED_10_FEATURES).issubset(tmp.columns):
            df_input = tmp
            loaded_csv = p
            break

if df_input is None and IN_COLAB:
    from google.colab import files
    print("Sube 'producto1_dataset_curado.csv' para ejecutar la inferencia batch:")
    up = files.upload()
    loaded_csv = next(iter(up.keys()))
    df_input = pd.read_csv(loaded_csv)

assert df_input is not None, "No se encontró el CSV de entrada con las 10 features."
print(f"Dataset cargado desde: {loaded_csv} ({df_input.shape[0]} propiedades)")

out_fam = predict_batch(df_input, profile="FAMILIA", published_price_col="costo_total_ars")
out_exec = predict_batch(df_input, profile="SOLTERO_EJECUTIVO", published_price_col="costo_total_ars")

resumen_global = pd.DataFrame({
    "Metrica": [
        "Propiedades evaluadas",
        "Score promedio (0-100)",
        "Oportunidades con ahorro >= 10%",
        "Propiedades con sobreprecio >= 10%",
        "ALTAMENTE RECOMENDADA",
        "RECOMENDADA",
        "DESCARTAR / SOBREPRECIO ALTO",
    ],
    "FAMILIA": [
        len(out_fam),
        round(float(out_fam["decision_score_0_100"].mean()), 2),
        int((out_fam["pct_ahorro"] >= 10.0).sum()),
        int((out_fam["pct_sobreprecio"] >= 10.0).sum()),
        int((out_fam["recomendacion"] == "ALTAMENTE RECOMENDADA").sum()),
        int((out_fam["recomendacion"] == "RECOMENDADA").sum()),
        int((out_fam["recomendacion"] == "DESCARTAR / SOBREPRECIO ALTO").sum()),
    ],
    "SOLTERO_EJECUTIVO": [
        len(out_exec),
        round(float(out_exec["decision_score_0_100"].mean()), 2),
        int((out_exec["pct_ahorro"] >= 10.0).sum()),
        int((out_exec["pct_sobreprecio"] >= 10.0).sum()),
        int((out_exec["recomendacion"] == "ALTAMENTE RECOMENDADA").sum()),
        int((out_exec["recomendacion"] == "RECOMENDADA").sum()),
        int((out_exec["recomendacion"] == "DESCARTAR / SOBREPRECIO ALTO").sum()),
    ],
})
display(resumen_global)

cols_show = ["m2_totales", "ambientes", "apto_mascotas", "score_amenities", "ruido_db_promedio", "cluster_label", "costo_total_ars", "precio_justo_ars", "pct_ahorro", "decision_score_0_100", "recomendacion"]
print("\nTop 5 Oportunidades · Perfil FAMILIA:")
display(out_fam.sort_values("decision_score_0_100", ascending=False)[cols_show].head(5))

print("\nTop 5 Oportunidades · Perfil SOLTERO_EJECUTIVO:")
display(out_exec.sort_values("decision_score_0_100", ascending=False)[cols_show].head(5))

## 5. Visualización de Resultados para Presentación

Graficamos:
1. **Costo Publicado vs. Precio Justo Teórico (`XGBoost`):** los puntos debajo de la diagonal representan oportunidades con ahorro real; por encima, sobreprecio.
2. **Distribución del Score de Decisión (`0–100`)** comparando `FAMILIA` vs. `SOLTERO_EJECUTIVO` y el **conteo de propiedades por Clúster de Habitabilidad**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Panel 1: Publicado vs Precio Justo
pub_m = out_fam["costo_total_ars"] / 1e6
pred_m = out_fam["precio_justo_ars"] / 1e6
limit_m = max(pub_m.quantile(0.98), pred_m.quantile(0.98))
sc = axes[0].scatter(pred_m, pub_m, c=out_fam["pct_ahorro"].clip(-30, 30), cmap="RdYlGn", s=16, alpha=0.75)
axes[0].plot([0, limit_m], [0, limit_m], "k--", linewidth=1.2, label="Precio Justo (y=x)")
axes[0].set_xlim(0, limit_m)
axes[0].set_ylim(0, limit_m)
axes[0].set_title("Costo Publicado vs. Precio Justo (Millones ARS)")
axes[0].set_xlabel("Precio Justo XGBoost (M ARS)")
axes[0].set_ylabel("Costo Publicado (M ARS)")
axes[0].legend()
fig.colorbar(sc, ax=axes[0], label="% Ahorro")

# Panel 2: Distribución del Score por Perfil
sns.kdeplot(out_fam["decision_score_0_100"], fill=True, ax=axes[1], label="FAMILIA", color="#1f77b4")
sns.kdeplot(out_exec["decision_score_0_100"], fill=True, ax=axes[1], label="SOLTERO_EJECUTIVO", color="#ff7f0e")
axes[1].axvline(75, color="green", linestyle="--", linewidth=1, label="Umbral Alt. Recom. (75)")
axes[1].set_title("Distribución del Score de Decisión (0–100)")
axes[1].set_xlabel("Score (0–100)")
axes[1].legend()

# Panel 3: Recomendaciones por Clúster (FAMILIA)
ct_plot = pd.crosstab(out_fam["cluster_label"], out_fam["recomendacion"])
ct_plot.plot(kind="barh", stacked=True, ax=axes[2], colormap="Set2")
axes[2].set_title("Recomendaciones por Clúster (FAMILIA)")
axes[2].set_xlabel("Cantidad de propiedades")
axes[2].set_ylabel("")
axes[2].legend(loc="lower right", fontsize=8)

fig.tight_layout()
plt.show()

## 6. Exportación de Resultados Consolidados (Handoff Producto 2)

Combinamos las evaluaciones de ambos perfiles en un único archivo tabular (`resultados_inferencia_caba.csv`) listo para conectar al Dashboard o automatización Make/n8n.

In [ ]:
df_export = df_input.copy()
df_export["cluster_id"] = out_fam["cluster_id"]
df_export["cluster_label"] = out_fam["cluster_label"]
df_export["precio_justo_ars"] = out_fam["precio_justo_ars"]
df_export["pct_sobreprecio"] = out_fam["pct_sobreprecio"]
df_export["pct_ahorro"] = out_fam["pct_ahorro"]
df_export["score_familia_0_100"] = out_fam["decision_score_0_100"]
df_export["recomendacion_familia"] = out_fam["recomendacion"]
df_export["score_soltero_0_100"] = out_exec["decision_score_0_100"]
df_export["recomendacion_soltero"] = out_exec["recomendacion"]

output_csv = PRODUCTO2_DIR / "resultados_inferencia_caba.csv"
PRODUCTO2_DIR.mkdir(parents=True, exist_ok=True)
df_export.to_csv(output_csv, index=False, encoding="utf-8")

print(f"Resultados exportados en: {output_csv} ({df_export.shape[0]} filas x {df_export.shape[1]} columnas)")
display(df_export[["cluster_label", "costo_total_ars", "precio_justo_ars", "pct_ahorro", "score_familia_0_100", "recomendacion_familia", "score_soltero_0_100", "recomendacion_soltero"]].head(5))